In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import joblib

In [2]:
DATA_PATH=Path('../data/processed/transport_processed.csv')
df=pd.read_csv(DATA_PATH)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 29 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   actual_arrival_delay_min   2000 non-null   int64  
 1   temperature_C              2000 non-null   float64
 2   humidity_percent           2000 non-null   int64  
 3   wind_speed_kmh             2000 non-null   int64  
 4   precipitation_mm           2000 non-null   float64
 5   event_attendance_est       2000 non-null   int64  
 6   traffic_congestion_index   2000 non-null   int64  
 7   holiday                    2000 non-null   int64  
 8   peak_hour                  2000 non-null   int64  
 9   weekday                    2000 non-null   int64  
 10  delayed                    2000 non-null   int64  
 11  hour                       2000 non-null   int64  
 12  scheduled_travel_time_min  2000 non-null   float64
 13  transport_type_Metro       2000 non-null   bool   
 14  tra

In [4]:
df.describe()

,actual_arrival_delay_min,temperature_C,humidity_percent,wind_speed_kmh,precipitation_mm,event_attendance_est,traffic_congestion_index,holiday,peak_hour,weekday,delayed,hour,scheduled_travel_time_min
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,13.318000,15.121350,64.714000,29.300500,9.860700,6420.250000,50.244000,0.089500,0.272000,2.976000,0.749500,11.572000,37.003500
std,9.289727,11.479424,20.334747,17.264015,5.781373,15198.306129,29.225751,0.285535,0.445101,1.990328,0.433409,6.903669,11.341243
min,-3.000000,-5.000000,30.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,16.000000
25%,5.000000,5.100000,46.000000,15.000000,4.900000,0.000000,25.000000,0.000000,0.000000,1.000000,0.000000,6.000000,27.000000
50%,13.000000,15.300000,65.000000,29.000000,9.700000,0.000000,50.000000,0.000000,0.000000,3.000000,1.000000,12.000000,37.000000
75%,21.000000,24.800000,83.000000,45.000000,14.800000,2000.000000,76.000000,0.000000,1.000000,5.000000,1.000000,18.000000,47.000000
max,29.000000,35.000000,99.000000,59.000000,20.000000,50000.000000,99.000000,1.000000,1.000000,6.000000,1.000000,23.000000,59.000000


In [5]:
df.select_dtypes(include='number')


,actual_arrival_delay_min,temperature_C,humidity_percent,wind_speed_kmh,precipitation_mm,event_attendance_est,traffic_congestion_index,holiday,peak_hour,weekday,delayed,hour,scheduled_travel_time_min
0,3,5.1,52,46,13.0,500,81,0,1,6,0,5,53.0
1,9,34.0,64,11,11.4,0,53,0,0,6,1,5,39.0
2,0,29.5,35,31,14.1,0,67,1,0,6,0,5,44.0
3,10,27.4,55,41,6.4,500,84,0,0,6,1,5,19.0
4,14,0.1,90,30,18.5,500,46,0,0,6,1,6,35.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,14,17.2,98,35,4.6,0,96,0,0,5,1,23,25.0
1996,25,0.0,89,44,15.4,0,12,0,1,6,1,0,38.0
1997,21,12.9,95,32,2.7,0,24,1,0,6,1,0,17.0
1998,9,17.8,55,35,8.8,2000,23,0,0,6,1,0,44.0


In [6]:
print(df.select_dtypes(include='number').columns.tolist())
print(df.select_dtypes(include='bool').columns.tolist())

['actual_arrival_delay_min', 'temperature_C', 'humidity_percent', 'wind_speed_kmh', 'precipitation_mm', 'event_attendance_est', 'traffic_congestion_index', 'holiday', 'peak_hour', 'weekday', 'delayed', 'hour', 'scheduled_travel_time_min']
['transport_type_Metro', 'transport_type_Train', 'transport_type_Tram', 'weather_condition_Cloudy', 'weather_condition_Fog', 'weather_condition_Rain', 'weather_condition_Snow', 'weather_condition_Storm', 'event_type_Festival', 'event_type_No events', 'event_type_Parade', 'event_type_Protest', 'event_type_Sports', 'season_Spring', 'season_Summer', 'season_Winter']


In [7]:
X = df.select_dtypes(include=['number', 'bool']).drop(columns=['actual_arrival_delay_min', 'delayed'])

y_reg = df['actual_arrival_delay_min']

In [8]:
X_train, X_test, y_train_class, y_test_class=train_test_split(X,y_reg,test_size=0.2,random_state=42)
y_train_reg = y_reg.loc[X_train.index]
y_test_reg  = y_reg.loc[X_test.index]

# Save split indices to avoid data leakage in evaluation notebook
Path('../models').mkdir(exist_ok=True)
np.save('../models/train_idx.npy', X_train.index.to_numpy())
np.save('../models/test_idx.npy',  X_test.index.to_numpy())

print(f'Train size: {len(X_train)}  |  Test size: {len(X_test)}')

Train size: 1600  |  Test size: 400


In [9]:
rfg=RandomForestRegressor(n_estimators=200, max_depth=None, random_state=42, n_jobs=-1,)
rfg.fit(X_train, y_train_reg)
y_pred_reg = rfg.predict(X_test)

mae  = mean_absolute_error(y_test_reg, y_pred_reg)
rmse = mean_squared_error(y_test_reg, y_pred_reg) ** 0.5

print(f'MAE:  {mae:.2f} min')
print(f'RMSE: {rmse:.2f} min')

MAE:  7.96 min
RMSE: 9.48 min


In [10]:
joblib.dump(rfg, '../models/rfg_model.pkl')

['../models/rfg_model.pkl']